# 🧠 Üretken Yapay Zeka — Derin Üretken Modeller
## VAE · GAN · GMMN · Difüzyon
### 3. Ders Uygulamalı Notebook

---
Bu notebook, derste işlenen teorik konuların Python ile uygulamalı gösterimidir.

**İçindekiler:**
1. Olasılıksal Çerçeve & MLE
2. KL Iraksaması
3. Latent Uzay & Manifold
4. ELBO Türetimi
5. VAE Implementasyonu (MNIST)
6. GAN Implementasyonu
7. GMMN & MMD
8. Difüzyon Modeli (DDPM)
9. Model Karşılaştırması & FID

In [ ]:
!pip install torch torchvision matplotlib numpy scipy scikit-learn tqdm -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import linalg
from sklearn.decomposition import PCA
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Seed sabitleme — tekrarlanabilirlik için
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Kullanılan cihaz: {device}')
print(f'PyTorch versiyonu: {torch.__version__}')

---
## Bölüm 1: Olasılıksal Çerçeve & Maksimum Olabilirlik (MLE)

Üretken modellerin temel hedefi:
$$\theta^* = \arg\max_{\theta} \mathbb{E}_{x \sim p_{\text{data}}}[\log p_{\text{model}}(x; \theta)]$$

Bu bölümde gerçek bir dağılımı (pdata) Gaussian karışımıyla temsil edip MLE ile parametrelerini öğreneceğiz.

In [ ]:
# -----------------------------------------------------------
# 1.1 — Gerçek veri dağılımını simüle et (2D Gaussian Karışımı)
# -----------------------------------------------------------
def sample_pdata(n=1000):
    """Gerçek veri dağılımını simüle eden 2D Gaussian karışımı."""
    centers = [(-3, 0), (3, 0), (0, 3)]
    weights = [0.4, 0.3, 0.3]
    samples = []
    for _ in range(n):
        center = centers[np.random.choice(len(centers), p=weights)]
        x = np.random.randn(2) * 0.8 + np.array(center)
        samples.append(x)
    return np.array(samples)

real_data = sample_pdata(1000)

# -----------------------------------------------------------
# 1.2 — Gaussian log-olasılık fonksiyonu
# -----------------------------------------------------------
def log_gaussian(x, mu, sigma):
    """N(mu, sigma^2 * I) altında x'in log-olasılığı."""
    d = x.shape[-1]
    diff = x - mu
    return -0.5 * (d * np.log(2 * np.pi) + d * np.log(sigma**2) +
                   np.sum(diff**2, axis=-1) / sigma**2)

# -----------------------------------------------------------
# 1.3 — MLE: Ortalama ve standart sapmayı veri üzerinden öğren
# -----------------------------------------------------------
mu_mle    = real_data.mean(axis=0)
sigma_mle = real_data.std()

log_likelihoods_mle = log_gaussian(real_data, mu_mle, sigma_mle)
log_likelihoods_bad = log_gaussian(real_data, np.array([0, 0]), 5.0)

print(f'MLE Parametreleri  → mu: {mu_mle.round(3)}, sigma: {sigma_mle:.3f}')
print(f'Ort. Log-Olasılık (MLE modeli): {log_likelihoods_mle.mean():.3f}')
print(f'Ort. Log-Olasılık (kötü model): {log_likelihoods_bad.mean():.3f}')
print('✅ MLE modeli daha yüksek log-olasılık üretiyor — beklenen sonuç!')

# -----------------------------------------------------------
# 1.4 — Görselleştirme
# -----------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('MLE: Gerçek Veri vs. Model Dağılımları', fontsize=14, fontweight='bold')

axes[0].scatter(real_data[:,0], real_data[:,1], alpha=0.4, s=15, c='steelblue')
axes[0].set_title('Gerçek Veri (pdata)\n3 Gaussian Karışımı', fontsize=11)
axes[0].set_xlim(-7, 7); axes[0].set_ylim(-4, 6)
axes[0].grid(True, alpha=0.3)

x_range = np.linspace(-7, 7, 100)
y_range = np.linspace(-4, 6, 100)
X, Y    = np.meshgrid(x_range, y_range)
grid    = np.stack([X.ravel(), Y.ravel()], axis=1)

Z_mle = np.exp(log_gaussian(grid, mu_mle, sigma_mle)).reshape(100, 100)
axes[1].contourf(X, Y, Z_mle, levels=20, cmap='Blues')
axes[1].scatter(real_data[:,0], real_data[:,1], alpha=0.2, s=10, c='red')
axes[1].set_title(f'MLE Modeli\nmu≈{mu_mle.round(1)}, σ={sigma_mle:.2f}', fontsize=11)
axes[1].grid(True, alpha=0.3)

Z_bad = np.exp(log_gaussian(grid, np.array([0,0]), 5.0)).reshape(100, 100)
axes[2].contourf(X, Y, Z_bad, levels=20, cmap='Oranges')
axes[2].scatter(real_data[:,0], real_data[:,1], alpha=0.2, s=10, c='red')
axes[2].set_title('Kötü Model\nmu=[0,0], σ=5.0', fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Bölüm 2: KL Iraksaması — MLE ≡ KL Minimizasyonu

Derste gösterildiği üzere:
$$D_{KL}(p_{\text{data}} \| p_{\text{model}}) \geq 0$$

MLE maksimizasyonu = KL ıraksaması minimizasyonu

In [ ]:
# -----------------------------------------------------------
# 2.1 — KL Iraksaması: Gaussian örneği
# KL(N(mu1,s1) || N(mu2,s2)) kapalı form formülü
# -----------------------------------------------------------
def kl_gaussian(mu1, s1, mu2, s2):
    """KL(N(mu1,s1^2) || N(mu2,s2^2)) — kapalı form."""
    return np.log(s2/s1) + (s1**2 + (mu1-mu2)**2) / (2*s2**2) - 0.5

# -----------------------------------------------------------
# 2.2 — KL'nin asimetrikliğini göster
# -----------------------------------------------------------
mu_vals    = np.linspace(-3, 3, 200)
kl_forward  = [kl_gaussian(m, 1.0, 0.0, 1.0) for m in mu_vals]
kl_backward = [kl_gaussian(0.0, 1.0, m, 1.0) for m in mu_vals]

kl_same = kl_gaussian(1.0, 2.0, 1.0, 2.0)
print(f'KL(N(1,2) || N(1,2)) = {kl_same:.6f}  ← sıfır olmalı ✅')
print(f'KL(N(0,1) || N(2,1)) = {kl_gaussian(0,1,2,1):.4f}')
print(f'KL(N(2,1) || N(0,1)) = {kl_gaussian(2,1,0,1):.4f}  ← asimetri!')

# -----------------------------------------------------------
# 2.3 — Görselleştirme
# -----------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('KL Iraksaması', fontsize=14, fontweight='bold')

axes[0].plot(mu_vals, kl_forward,  'b-', lw=2, label='KL(N(μ,1) ‖ N(0,1))')
axes[0].plot(mu_vals, kl_backward, 'r--', lw=2, label='KL(N(0,1) ‖ N(μ,1))')
axes[0].axvline(0, color='k', ls=':', alpha=0.5)
axes[0].axhline(0, color='k', ls=':', alpha=0.5)
axes[0].set_xlabel('μ (ortalama kayması)')
axes[0].set_ylabel('KL Iraksaması')
axes[0].set_title('KL Asimetrisi\nKL(p‖q) ≠ KL(q‖p)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

x = np.linspace(-6, 6, 300)
p = np.exp(-0.5*x**2) / np.sqrt(2*np.pi)
q = np.exp(-0.5*(x-2)**2) / np.sqrt(2*np.pi)
axes[1].fill_between(x, p, alpha=0.4, color='blue', label='p = N(0,1)')
axes[1].fill_between(x, q, alpha=0.4, color='red',  label='q = N(2,1)')
axes[1].set_xlabel('x')
axes[1].set_ylabel('Olasılık Yoğunluğu')
axes[1].set_title(f'Örnek: KL(p‖q) = {kl_gaussian(0,1,2,1):.3f}\n'
                  f'Üretken modelde q → p\'ye yaklaşmalı')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Bölüm 3: Latent Uzay & Manifold Hipotezi

**Manifold Hipotezi:** Yüksek boyutlu doğal veri, düşük boyutlu bir manifold üzerinde yoğunlaşır.

MNIST rakamları: 784 piksel → ~10 boyutlu manifold

In [ ]:
# -----------------------------------------------------------
# 3.1 — MNIST yükleme  (sklearn import üstte yapıldı)
# -----------------------------------------------------------
transform = transforms.Compose([transforms.ToTensor()])
mnist_full = datasets.MNIST('./data', train=True, download=True, transform=transform)

# 2000 örnek ile çalış (hız için)
subset_idx = torch.randperm(len(mnist_full))[:2000]
X_raw = mnist_full.data[subset_idx].numpy().reshape(2000, -1) / 255.0
y_raw = mnist_full.targets[subset_idx].numpy()
print(f'Ham veri boyutu: {X_raw.shape}  ← 784 piksel/rakam')

# -----------------------------------------------------------
# 3.2 — PCA ile boyut indirgeme: varyans analizi
# -----------------------------------------------------------
pca_full = PCA(n_components=100)
pca_full.fit(X_raw)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

n90 = np.searchsorted(cumvar, 0.90) + 1
n95 = np.searchsorted(cumvar, 0.95) + 1
print(f'%90 varyans için gereken bileşen sayısı: {n90}')
print(f'%95 varyans için gereken bileşen sayısı: {n95}')
print(f'→ 784 pikseli sadece ~{n90} bileşenle temsil edebiliyoruz!')

# -----------------------------------------------------------
# 3.3 — 2D PCA projeksiyonu
# -----------------------------------------------------------
pca2     = PCA(n_components=2)
X_pca2   = pca2.fit_transform(X_raw)

# -----------------------------------------------------------
# 3.4 — Görselleştirme
# -----------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Manifold Hipotezi — MNIST Üzerinde Gösterim', fontsize=14, fontweight='bold')

axes[0].plot(range(1, 101), cumvar, 'b-', lw=2)
axes[0].axhline(0.90, color='orange', ls='--', label=f'%90 → {n90} bileşen')
axes[0].axhline(0.95, color='red',    ls='--', label=f'%95 → {n95} bileşen')
axes[0].fill_between(range(1, 101), cumvar, alpha=0.2)
axes[0].set_xlabel('PCA Bileşen Sayısı')
axes[0].set_ylabel('Birikimli Açıklanan Varyans')
axes[0].set_title(f'784 Piksel → Manifold Boyutu\n'
                  f'Veri esasen {n90}D manifold üzerinde!')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

colors = plt.cm.tab10(np.linspace(0, 1, 10))
for digit in range(10):
    mask = y_raw == digit
    axes[1].scatter(X_pca2[mask,0], X_pca2[mask,1],
                    c=[colors[digit]], alpha=0.5, s=15, label=str(digit))
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].set_title('MNIST — 2D PCA Projeksiyonu\nLatent uzayda rakamlar kümeleniyor')
axes[1].legend(title='Rakam', loc='upper right', ncol=2, fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------
# 3.5 — Latent Uzay Aritmetiği: z(7) − z(1) + z(0) ≈ z(6) ?
# -----------------------------------------------------------
def get_class_centroid(X_latent, y, cls):
    """Belirli bir sınıfın latent uzaydaki ağırlık merkezi."""
    return X_latent[y == cls].mean(axis=0)

pca10    = PCA(n_components=10)
X_pca10  = pca10.fit_transform(X_raw)
centroids = {d: get_class_centroid(X_pca10, y_raw, d) for d in range(10)}

z_arith   = centroids[7] - centroids[1] + centroids[0]
distances = {d: np.linalg.norm(z_arith - centroids[d]) for d in range(10)}
closest   = min(distances, key=distances.get)

print('Latent Uzay Aritmetiği Örneği')
print('=' * 40)
print('z(7) − z(1) + z(0) = ?')
print(f'Mesafeler: {dict(sorted(distances.items(), key=lambda x: x[1])[:5])}')
print(f'En yakın rakam: {closest}')
print(f'(Beklenen: 6 = 7−1+0)')

fig, ax = plt.subplots(figsize=(8, 4))
digits = list(distances.keys())
dists  = [distances[d] for d in digits]
ax.bar(digits, dists, color=['red' if d==closest else 'steelblue' for d in digits])
ax.set_xlabel('Rakam')
ax.set_ylabel('Öklid Mesafesi')
ax.set_title('z(7) − z(1) + z(0) → Hangi rakama en yakın?\n(Kırmızı = en yakın)')
ax.set_xticks(digits)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
## Bölüm 4: ELBO — Variasyonel Alt Sınır

$$\log p(x) \geq \underbrace{\mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)]}_{\text{Yeniden Yapılandırma}} - \underbrace{D_{KL}(q_\phi(z|x) \| p(z))}_{\text{Düzenlileştirme}} =: \mathcal{L}(x)$$

ELBO, gerçek log-olasılığın bir **alt sınırı**dır. Bunu aşağıda görselleştireceğiz.

In [ ]:
# -----------------------------------------------------------
# 4.1 — ELBO'nun Gaussian için kapalı form hesabı
# -----------------------------------------------------------
def kl_standard_gaussian(mu, log_var):
    """
    KL(N(mu, diag(sigma^2)) || N(0,I)) — VAE'de kullanılan kapalı form
    = -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    """
    return -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp(), dim=-1)

# -----------------------------------------------------------
# 4.2 — ELBO bileşenlerini analiz et
# mu_range: x ekseni, log_var_range: y ekseni
# KL shape: (len(mu_range), len(log_var_range)) → contourf için .T gerekiyor
# -----------------------------------------------------------
mu_range      = torch.linspace(-3, 3, 100)
log_var_range = torch.linspace(-3, 1, 100)

MU, LV = torch.meshgrid(mu_range, log_var_range, indexing='ij')
KL = kl_standard_gaussian(
    MU.reshape(-1, 1), LV.reshape(-1, 1)
).reshape(100, 100)   # shape: (mu_dim=100, lv_dim=100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ELBO Bileşenleri', fontsize=14, fontweight='bold')

# KL ısı haritası — contourf(x_ekseni, y_ekseni, Z) → Z.T ile transpoze
im = axes[0].contourf(mu_range.numpy(), log_var_range.numpy(),
                      KL.numpy().T, levels=30, cmap='RdYlGn_r')
plt.colorbar(im, ax=axes[0], label='KL Değeri')
axes[0].axvline(0, color='white', ls='--', lw=2, label='mu=0')
axes[0].axhline(0, color='cyan',  ls='--', lw=2, label='log_var=0 (sigma=1)')
axes[0].set_xlabel('mu (posterior ortalama)')
axes[0].set_ylabel('log_var (posterior log varyans)')
axes[0].set_title("KL Düzenlileştirme Terimi\nYeşil = KL≈0 (prior'a yakın)")
axes[0].legend(fontsize=9)

# ELBO dengesi — farklı eğitim aşamalarında kayıp bileşenlerini göster
recon_losses_ex = np.array([0.1, 0.5, 1.0, 2.0, 5.0])
kl_losses_ex    = np.array([3.0, 2.0, 1.5, 1.0, 0.2])

x_ticks = range(len(recon_losses_ex))
width   = 0.3
axes[1].bar([x - width/2 for x in x_ticks], recon_losses_ex, width,
            label='Rekonstrüksiyon Kaybı', color='steelblue', alpha=0.8)
axes[1].bar([x + width/2 for x in x_ticks], kl_losses_ex, width,
            label='KL Düzenlileştirme', color='salmon', alpha=0.8)
axes[1].plot(x_ticks, recon_losses_ex + kl_losses_ex, 'ko-', ms=8,
             label='Toplam Kayıp (−ELBO)', lw=2)
axes[1].set_xlabel('Eğitim Aşaması (Erken → Geç)')
axes[1].set_ylabel('Kayıp')
axes[1].set_title("ELBO'nun İki Terimi Arasındaki Denge\n(β=1 durumu)")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('ELBO Yorumu:')
print("• Rekonstrüksiyon ↓ → decoder kaliteli görüntü üretiyor")
print("• KL ↓            → latent uzay prior'a yakın (düzenli)")
print('• Bu iki terim çatışır → denge noktası ELBO maksimumudur')

---
## Bölüm 5: Variational Autoencoder (VAE)

**Reparametrization Trick:**
$$z = \mu_\phi(x) + \sigma_\phi(x) \odot \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, I)$$

**Kayıp Fonksiyonu:**
$$\mathcal{L} = \|x - \hat{x}\|^2 + \beta \cdot D_{KL}(q_\phi \| p)$$

In [ ]:
# -----------------------------------------------------------
# 5.1 — VAE Mimarisi
# -----------------------------------------------------------
class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, latent_dim=2):
        super().__init__()
        self.latent_dim = latent_dim

        # Encoder: x → (mu, log_var)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU()
        )
        self.fc_mu      = nn.Linear(hidden_dim, latent_dim)
        self.fc_log_var = nn.Linear(hidden_dim, latent_dim)

        # Decoder: z → x̂
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()   # [0,1] aralığına sıkıştır (BCE kaybı için)
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_log_var(h)

    def reparametrize(self, mu, log_var):
        """Reparametrization Trick: z = mu + sigma * eps"""
        if self.training:
            std = torch.exp(0.5 * log_var)
            eps = torch.randn_like(std)   # eps ~ N(0,I)
            return mu + std * eps          # gradyan mu ve log_var üzerinden akar!
        return mu   # inference sırasında ortalamayı kullan

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z           = self.reparametrize(mu, log_var)
        x_hat       = self.decode(z)
        return x_hat, mu, log_var

def vae_loss(x, x_hat, mu, log_var, beta=1.0):
    """ELBO kaybı = Rekonstrüksiyon (BCE) + β * KL"""
    recon = F.binary_cross_entropy(x_hat, x, reduction='sum') / x.size(0)
    kl    = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / x.size(0)
    return recon + beta * kl, recon, kl

print('VAE Mimarisi:')
vae_demo = VAE(latent_dim=2)
print(vae_demo)
total_params = sum(p.numel() for p in vae_demo.parameters())
print(f'\nToplam parametre sayısı: {total_params:,}')

In [ ]:
# -----------------------------------------------------------
# 5.2 — Reparametrization Trick: gradyan akışı gösterimi
# -----------------------------------------------------------
print('Reparametrization Trick Neden Gerekli?')
print('=' * 50)
print()
print('SORUN: Stokastik örnekleme türevlenemez!')
print('  z ~ N(mu, sigma^2)  →  dL/dmu = ??  (backprop ÇALIŞMAZ)')
print()
print('ÇÖZÜM: Gürültüyü dışarı çıkar!')
print('  eps ~ N(0,I)        →  sabit gürültü, parametreye bağlı DEĞİL')
print('  z = mu + sigma*eps  →  mu ve sigma üzerinden türev ÇALIŞIR!')
print()

mu_ex = torch.tensor([1.0, -0.5], requires_grad=True)
lv_ex = torch.tensor([0.2, -0.3], requires_grad=True)

std  = torch.exp(0.5 * lv_ex)
eps  = torch.randn(2)   # sabit gürültü
z    = mu_ex + std * eps
loss = z.sum()
loss.backward()

print(f'Örnek z değerleri : {z.detach().numpy().round(3)}')
print(f'dL/dmu gradyanı   : {mu_ex.grad.numpy().round(3)}  ← backprop çalıştı! ✅')
print(f'dL/dlog_var gradyan: {lv_ex.grad.numpy().round(3)}')

In [ ]:
# -----------------------------------------------------------
# 5.3 — VAE Eğitimi (MNIST, hızlı versiyon)
# -----------------------------------------------------------
LATENT_DIM = 2
BATCH_SIZE  = 256
EPOCHS      = 10
BETA        = 1.0

# [0,1] normalize veri — BCE kaybıyla uyumlu
train_loader = DataLoader(
    datasets.MNIST('./data', train=True, transform=transforms.ToTensor()),
    batch_size=BATCH_SIZE, shuffle=True
)

vae           = VAE(latent_dim=LATENT_DIM).to(device)
optimizer_vae = optim.Adam(vae.parameters(), lr=1e-3)

train_losses = []
recon_losses = []
kl_losses    = []

for epoch in range(EPOCHS):
    vae.train()
    total_loss = recon_tot = kl_tot = 0

    for x, _ in train_loader:
        x = x.view(-1, 784).to(device)
        x_hat, mu, log_var = vae(x)
        loss, recon, kl    = vae_loss(x, x_hat, mu, log_var, BETA)

        optimizer_vae.zero_grad()
        loss.backward()
        optimizer_vae.step()

        total_loss += loss.item()
        recon_tot  += recon.item()
        kl_tot     += kl.item()

    n = len(train_loader)
    train_losses.append(total_loss / n)
    recon_losses.append(recon_tot / n)
    kl_losses.append(kl_tot / n)
    print(f'Epoch {epoch+1:2d}/{EPOCHS} | '
          f'Toplam: {total_loss/n:.2f} | '
          f'Rekon: {recon_tot/n:.2f} | '
          f'KL: {kl_tot/n:.3f}')

print('\nVAE eğitimi tamamlandı! ✅')

In [ ]:
# -----------------------------------------------------------
# 5.4 — Eğitim Eğrileri & Sonuçlar
# DÜZELTME: grid_img dead-code kaldırıldı; combined.detach() eklendi;
#           add_axes/tight_layout çakışması giderildi (ayrı figüre taşındı)
# -----------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('VAE Sonuçları', fontsize=15, fontweight='bold')

# ── Kayıp eğrileri ──────────────────────────────────────────
axes[0,0].plot(train_losses, 'b-o', ms=5, label='Toplam')
axes[0,0].plot(recon_losses, 'r--s', ms=5, label='Rekonstrüksiyon')
axes[0,0].plot(kl_losses,    'g:^',  ms=5, label='KL')
axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Kayıp')
axes[0,0].set_title('Eğitim Kayıp Eğrileri')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# ── 2D Latent uzay görselleştirme ───────────────────────────
vae.eval()
test_ds     = datasets.MNIST('./data', train=False, transform=transforms.ToTensor())
test_loader = DataLoader(test_ds, batch_size=2000, shuffle=False)
x_test, y_test = next(iter(test_loader))
x_test_flat    = x_test.view(-1, 784).to(device)

with torch.no_grad():
    mu_test, _ = vae.encode(x_test_flat)
    mu_np = mu_test.cpu().numpy()
    y_np  = y_test.numpy()

scatter = axes[0,1].scatter(mu_np[:,0], mu_np[:,1],
                             c=y_np, cmap='tab10', alpha=0.6, s=10)
plt.colorbar(scatter, ax=axes[0,1])
axes[0,1].set_xlabel('z₁'); axes[0,1].set_ylabel('z₂')
axes[0,1].set_title('2D Latent Uzay\nRenkler = Rakam sınıfları')
axes[0,1].grid(True, alpha=0.3)

# ── Orijinal vs. Yeniden yapılandırılmış ────────────────────
n_show = 8
with torch.no_grad():
    x_show     = x_test[:n_show].view(-1, 784).to(device)
    x_rec, _, _ = vae(x_show)
    x_rec       = x_rec.cpu().view(-1, 28, 28)

orig_grid  = x_test[:n_show].squeeze()
recon_grid = x_rec

# combined Tensor (gradient yok, ama detach ile güvenli)
combined = torch.zeros(2*28, n_show*28)
for i in range(n_show):
    combined[:28,  i*28:(i+1)*28] = orig_grid[i]
    combined[28:,  i*28:(i+1)*28] = recon_grid[i]

axes[0,2].imshow(combined.detach().numpy(), cmap='gray')
axes[0,2].set_title("Orijinal (üst) vs. VAE Rekonstrüksiyon (alt)\nBulanıklık VAE'nin bilinen dezavantajı")
axes[0,2].axis('off')

# ── Latent uzay örnekleme (2D grid) ─────────────────────────
n_grid  = 15
z1      = np.linspace(-3, 3, n_grid)
z2      = np.linspace(-3, 3, n_grid)
canvas  = np.zeros((n_grid*28, n_grid*28))

vae.eval()
with torch.no_grad():
    for i, zi in enumerate(z1):
        for j, zj in enumerate(z2):
            z_sample = torch.tensor([[zi, zj]], dtype=torch.float32).to(device)
            img = vae.decode(z_sample).cpu().view(28, 28).numpy()
            canvas[(n_grid-1-j)*28:(n_grid-j)*28, i*28:(i+1)*28] = img

axes[1,0].imshow(canvas, cmap='gray')
axes[1,0].set_title(f'2D Latent Uzaydan Örnekleme\n{n_grid}×{n_grid} grid')
axes[1,0].set_xlabel('z₁'); axes[1,0].set_ylabel('z₂')

# ── β-VAE etkisi ────────────────────────────────────────────
beta_vals = [0.1, 1.0, 4.0]
colors_b  = ['blue', 'green', 'red']
for beta_val, color in zip(beta_vals, colors_b):
    vae_b = VAE(latent_dim=2).to(device)
    opt_b = optim.Adam(vae_b.parameters(), lr=1e-3)
    kl_hist = []
    for epoch in range(5):
        kl_ep = 0
        for x_b, _ in train_loader:
            x_b = x_b.view(-1, 784).to(device)
            xh, mu_b, lv_b         = vae_b(x_b)
            loss_b, _, kl_b        = vae_loss(x_b, xh, mu_b, lv_b, beta_val)
            opt_b.zero_grad(); loss_b.backward(); opt_b.step()
            kl_ep += kl_b.item()
        kl_hist.append(kl_ep / len(train_loader))
    axes[1,1].plot(kl_hist, color=color, marker='o', ms=5, label=f'β={beta_val}')

axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('KL Kaybı')
axes[1,1].set_title('β-VAE: β Değerinin KL\'ye Etkisi\nβ büyük → daha sıkı düzenlileştirme')
axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

# ── Posterior Collapse açıklaması ───────────────────────────
axes[1,2].text(0.5, 0.62, 'Posterior Collapse',
               ha='center', va='center', fontsize=14, fontweight='bold',
               transform=axes[1,2].transAxes, color='red')
axes[1,2].text(0.5, 0.38,
               "Decoder x'i z'den bağımsız öğrenirse\n"
               'DKL → 0 ama latent uzay anlamsızlaşır.\n\n'
               'Çözüm: β > 1 (β-VAE),\n'
               'warm-up (KL ağırlığını yavaş artır)',
               ha='center', va='center', fontsize=11,
               transform=axes[1,2].transAxes,
               bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
axes[1,2].axis('off')
axes[1,2].set_title('Posterior Collapse Sorunu')

plt.tight_layout()
plt.show()

---
## Bölüm 6: Generative Adversarial Networks (GAN)

**Minimax Oyun:**
$$\min_G \max_D V(G,D) = \mathbb{E}_{x\sim p_{\text{data}}}[\log D(x)] + \mathbb{E}_{z\sim p_z}[\log(1-D(G(z)))]$$

**Nash Dengesinde:** $D^*(x) = \frac{p_{\text{data}}(x)}{p_{\text{data}}(x)+p_g(x)} = 0.5$

In [ ]:
# -----------------------------------------------------------
# 6.1 — GAN Mimarisi
# -----------------------------------------------------------
class Generator(nn.Module):
    """Üretici: z (gürültü) → sahte veri"""
    def __init__(self, z_dim=100, hidden=256, out_dim=784):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden),   nn.LeakyReLU(0.2),
            nn.BatchNorm1d(hidden),
            nn.Linear(hidden, hidden*2), nn.LeakyReLU(0.2),
            nn.BatchNorm1d(hidden*2),
            nn.Linear(hidden*2, out_dim),
            nn.Tanh()   # [-1,1] aralığı
        )
    def forward(self, z): return self.net(z)

class Discriminator(nn.Module):
    """Ayrımcı: veri → gerçek/sahte olasılığı"""
    def __init__(self, in_dim=784, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden*2),  nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(hidden*2, hidden),  nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(hidden, 1),
            nn.Sigmoid()   # [0,1] = gerçek olasılığı
        )
    def forward(self, x): return self.net(x)

# -----------------------------------------------------------
# 6.2 — Optimal Discriminator formülünü görselleştir
# D*(x) = p_data(x) / (p_data(x) + p_g(x))
# -----------------------------------------------------------
x_range   = np.linspace(-5, 5, 300)
p_data_ex = 0.6*np.exp(-0.5*(x_range-1)**2/0.5) + 0.4*np.exp(-0.5*(x_range+2)**2/0.8)
p_data_ex /= p_data_ex.max()
p_g_early  = np.exp(-0.5*(x_range+3)**2/1.5)
p_g_late   = 0.5*np.exp(-0.5*(x_range-1)**2/0.6) + 0.5*np.exp(-0.5*(x_range+2)**2/0.9)
p_g_late  /= p_g_late.max()

D_star_early = p_data_ex / (p_data_ex + p_g_early + 1e-8)
D_star_late  = p_data_ex / (p_data_ex + p_g_late  + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('GAN: Optimal Discriminator & Nash Dengesi', fontsize=14, fontweight='bold')

axes[0].fill_between(x_range, p_data_ex, alpha=0.4, color='blue', label='p_data')
axes[0].fill_between(x_range, p_g_early,  alpha=0.4, color='red',  label='p_g (kötü)')
axes[0].plot(x_range, D_star_early, 'k-', lw=2, label='D*(x)')
axes[0].axhline(0.5, color='gray', ls='--', alpha=0.7, label='0.5 (Nash)')
axes[0].set_title('Erken Eğitim\nD*(x) ≈1 gerçek bölgelerde, ≈0 sahte')
axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_ylim(-0.05, 1.2)

axes[1].fill_between(x_range, p_data_ex, alpha=0.4, color='blue', label='p_data')
axes[1].fill_between(x_range, p_g_late,   alpha=0.4, color='red',  label='p_g (iyi)')
axes[1].plot(x_range, D_star_late, 'k-', lw=2, label='D*(x)')
axes[1].axhline(0.5, color='gray', ls='--', alpha=0.7, label='D*=0.5 (Nash Dengesi)')
axes[1].set_title('Nash Dengesi\np_g ≈ p_data → D*(x) ≈ 0.5 her yerde')
axes[1].legend(); axes[1].grid(True, alpha=0.3); axes[1].set_ylim(-0.05, 1.2)

plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------
# 6.3 — GAN Eğitimi (MNIST)
# Not: Generator çıkışı Tanh → [-1,1]; veri de [-1,1] normalize edilmeli
# D adımında x_fake.detach() ile G dondurulur; G adımında yeniden üretilir
# -----------------------------------------------------------
Z_DIM      = 64
GAN_EPOCHS = 10

transform_gan = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))   # [0,1] → [-1,1]
])
gan_loader = DataLoader(
    datasets.MNIST('./data', train=True, transform=transform_gan),
    batch_size=256, shuffle=True
)

G         = Generator(z_dim=Z_DIM).to(device)
D_net     = Discriminator().to(device)
opt_G     = optim.Adam(G.parameters(),     lr=2e-4, betas=(0.5, 0.999))
opt_D     = optim.Adam(D_net.parameters(), lr=2e-4, betas=(0.5, 0.999))
criterion = nn.BCELoss()

G_losses, D_losses = [], []

for epoch in range(GAN_EPOCHS):
    G.train(); D_net.train()
    g_loss_ep = d_loss_ep = 0

    for x_real, _ in gan_loader:
        batch  = x_real.size(0)
        x_real = x_real.view(batch, -1).to(device)

        real_labels = torch.ones(batch, 1).to(device)
        fake_labels = torch.zeros(batch, 1).to(device)

        # ── Discriminator Adımı ──────────────────────────────
        z      = torch.randn(batch, Z_DIM).to(device)
        x_fake = G(z).detach()   # G parametreleri bu adımda dondurulur
        loss_D = (criterion(D_net(x_real), real_labels) +
                  criterion(D_net(x_fake), fake_labels))
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()

        # ── Generator Adımı ──────────────────────────────────
        z      = torch.randn(batch, Z_DIM).to(device)
        x_fake = G(z)            # Yeni örnekleme — gradient akar
        # G, D'yi yanıltmak istiyor → sahteye 'real' label atar
        loss_G = criterion(D_net(x_fake), real_labels)
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()

        g_loss_ep += loss_G.item()
        d_loss_ep += loss_D.item()

    n = len(gan_loader)
    G_losses.append(g_loss_ep / n)
    D_losses.append(d_loss_ep / n)
    print(f'Epoch {epoch+1:2d}/{GAN_EPOCHS} | G: {g_loss_ep/n:.4f} | D: {d_loss_ep/n:.4f}')

print('\nGAN eğitimi tamamlandı! ✅')

In [ ]:
# -----------------------------------------------------------
# 6.4 — GAN Sonuçları
# -----------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('GAN Sonuçları', fontsize=14, fontweight='bold')

axes[0].plot(G_losses, 'b-o', ms=5, label='Generator Kaybı')
axes[0].plot(D_losses, 'r-s', ms=5, label='Discriminator Kaybı')
axes[0].axhline(np.log(2), color='gray', ls='--', label=f'Teorik Nash = {np.log(2):.3f}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Kayıp')
axes[0].set_title("G ve D Kayıp Eğrileri\nNash dengesinde ikisi de ~0.693'e yaklaşır")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

G.eval()
with torch.no_grad():
    z_sample  = torch.randn(64, Z_DIM).to(device)
    fake_imgs = G(z_sample).cpu().view(-1, 28, 28)
    fake_imgs = (fake_imgs + 1) / 2   # [-1,1] → [0,1]

canvas_gan = np.zeros((8*28, 8*28))
for idx in range(64):
    r, c = idx // 8, idx % 8
    canvas_gan[r*28:(r+1)*28, c*28:(c+1)*28] = fake_imgs[idx].numpy()

axes[1].imshow(canvas_gan, cmap='gray')
axes[1].set_title('GAN — Üretilen Görüntüler (8×8 grid)')
axes[1].axis('off')

axes[2].text(0.5, 0.70, 'GAN Sorunları', ha='center', va='center',
             fontsize=14, fontweight='bold', transform=axes[2].transAxes, color='darkred')
problems = [
    '❌ Mod Çöküşü (Mode Collapse):\n   G sadece birkaç tip üretir',
    '❌ Eğitim Dengesizliği:\n   D çok iyi → G gradyanı yok',
    '❌ Vanishing Gradient:\n   log(1-D(G(z))) saturasyonu',
    '✅ Çözüm: WGAN, WGAN-GP,\n   Spectral Norm, Progressive GAN',
]
for i, prob in enumerate(problems):
    axes[2].text(0.05, 0.55 - i*0.14, prob,
                 ha='left', va='center', fontsize=10,
                 transform=axes[2].transAxes)
axes[2].axis('off')

plt.tight_layout()
plt.show()

---
## Bölüm 7: Generative Moment Matching Networks (GMMN) & MMD

**Maximum Mean Discrepancy:**
$$\text{MMD}^2(p,q) = \mathbb{E}_{x,x'}[k(x,x')] - 2\mathbb{E}_{x,y}[k(x,y)] + \mathbb{E}_{y,y'}[k(y,y')]$$

**Gauss Çekirdeği:** $k(x,x') = \exp(-\|x-x'\|^2/\sigma^2)$

In [ ]:
# -----------------------------------------------------------
# 7.1 — MMD hesaplama
# -----------------------------------------------------------
def gaussian_kernel(x, y, sigma=1.0):
    """Gauss (RBF) çekirdek matrisi: K[i,j] = exp(-||x_i - y_j||^2 / sigma^2)"""
    diff    = x.unsqueeze(1) - y.unsqueeze(0)    # (n, m, d)
    dist_sq = (diff**2).sum(-1)                   # (n, m)
    return torch.exp(-dist_sq / (sigma**2))

def mmd_loss(x, y, sigmas=[0.5, 1.0, 2.0]):
    """
    MMD^2 — çoklu çekirdek (multi-scale kernel).
    x: gerçek veri,  y: üretilen veri
    """
    mmd = 0.0
    for sigma in sigmas:
        Kxx = gaussian_kernel(x, x, sigma)
        Kxy = gaussian_kernel(x, y, sigma)
        Kyy = gaussian_kernel(y, y, sigma)
        mmd += Kxx.mean() - 2*Kxy.mean() + Kyy.mean()
    return mmd

# -----------------------------------------------------------
# 7.2 — MMD sezgisel test
# -----------------------------------------------------------
print('MMD Sezgisel Test')
print('=' * 45)
torch.manual_seed(0)

test_cases = [
    ('Aynı dağılım   N(0,1) vs N(0,1)',   (0, 1),   (0, 1)),
    ('Yakın dağılım  N(0,1) vs N(0.5,1)', (0, 1),   (0.5, 1)),
    ('Uzak dağılım   N(0,1) vs N(3,1)',   (0, 1),   (3, 1)),
    ('Farklı var.    N(0,1) vs N(0,2)',   (0, 1),   (0, 2)),
]

for name, (m1,s1), (m2,s2) in test_cases:
    p = torch.randn(500, 2) * s1 + m1
    q = torch.randn(500, 2) * s2 + m2
    print(f'{name}: MMD = {mmd_loss(p, q).item():.4f}')

In [ ]:
# -----------------------------------------------------------
# 7.3 — GMMN: Basit üretici (GAN'sız, discriminator'sız)
# -----------------------------------------------------------
class GMMN(nn.Module):
    """Sadece bir üretici ağ — MMD kaybıyla eğitilir."""
    def __init__(self, z_dim=10, hidden=128, out_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, out_dim)
        )
    def forward(self, z): return self.net(z)

def sample_target(n=256):
    centers = torch.tensor([[-3.,0.],[3.,0.],[0.,3.]])
    idx     = torch.randint(3, (n,))
    return centers[idx] + torch.randn(n, 2) * 0.8

gmmn      = GMMN(z_dim=8, hidden=128, out_dim=2).to(device)
opt_gmmn  = optim.Adam(gmmn.parameters(), lr=1e-3)

mmd_history = []
GMMN_STEPS  = 1000

for step in range(GMMN_STEPS):
    z         = torch.randn(256, 8).to(device)
    x_real_g  = sample_target(256).to(device)
    x_fake_g  = gmmn(z)
    loss_mmd  = mmd_loss(x_real_g, x_fake_g)

    opt_gmmn.zero_grad()
    loss_mmd.backward()
    opt_gmmn.step()

    if step % 100 == 0:
        mmd_history.append(loss_mmd.item())
        print(f'Adım {step:4d} | MMD = {loss_mmd.item():.6f}')

print('\nGMMN eğitimi tamamlandı! ✅')

In [ ]:
# -----------------------------------------------------------
# 7.4 — GMMN Sonuçları
# -----------------------------------------------------------
gmmn.eval()
with torch.no_grad():
    z_test        = torch.randn(1000, 8).to(device)
    generated_gmmn = gmmn(z_test).cpu().numpy()
real_gmmn = sample_target(1000).numpy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('GMMN Sonuçları', fontsize=14, fontweight='bold')

axes[0].scatter(real_gmmn[:,0], real_gmmn[:,1], alpha=0.4, s=10, c='steelblue')
axes[0].set_title('Gerçek Veri — 3 Gaussian Karışımı')
axes[0].set_xlim(-7,7); axes[0].set_ylim(-4,6); axes[0].grid(True, alpha=0.3)

axes[1].scatter(generated_gmmn[:,0], generated_gmmn[:,1], alpha=0.4, s=10, c='salmon')
axes[1].set_title('GMMN — Üretilen Veri\nSadece Generator + MMD Kaybı')
axes[1].set_xlim(-7,7); axes[1].set_ylim(-4,6); axes[1].grid(True, alpha=0.3)

axes[2].plot(range(0, GMMN_STEPS, 100), mmd_history, 'g-o', ms=6)
axes[2].set_xlabel('Adım'); axes[2].set_ylabel('MMD Kaybı')
axes[2].set_title('MMD Kaybı Azalıyor\n(Dağılımlar yaklaşıyor)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('VAE vs GAN vs GMMN — Temel Fark:')
print('• VAE:  Encoder + Decoder, ELBO kaybı')
print('• GAN:  Generator + Discriminator, minimax oyunu')
print('• GMMN: Sadece Generator, MMD (kernel) kaybı — daha basit ama kalite zorlu!')

---
## Bölüm 8: Difüzyon Modelleri (DDPM)

**İleri Süreç (Gürültü Ekleme):**
$$q(x_t|x_{t-1}) = \mathcal{N}(x_t; \sqrt{1-\beta_t}\,x_{t-1},\, \beta_t I)$$

**Ters Süreç (Gürültü Temizleme):**
$$p_\theta(x_{t-1}|x_t) = \mathcal{N}(x_{t-1}; \mu_\theta(x_t,t), \Sigma_\theta(x_t,t))$$

**Eğitim Kaybı:**
$$\mathcal{L} = \mathbb{E}_{t,x_0,\varepsilon}\left[\|\varepsilon - \varepsilon_\theta(x_t,t)\|^2\right]$$

In [ ]:
# -----------------------------------------------------------
# 8.1 — İleri Süreç: Gürültü Ekleme
# -----------------------------------------------------------
class DiffusionForwardProcess:
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02):
        self.T         = T
        self.beta      = torch.linspace(beta_start, beta_end, T)
        self.alpha     = 1 - self.beta
        self.alpha_bar = torch.cumprod(self.alpha, dim=0)  # ᾱ_t = Π α_s

    def q_sample(self, x0, t):
        """
        Kapalı form: x_t = sqrt(ᾱ_t)*x0 + sqrt(1-ᾱ_t)*eps
        Zincir tekrarına gerek yok — doğrudan t adımlık gürültü.
        """
        alpha_bar_t = self.alpha_bar[t]
        eps         = torch.randn_like(x0)
        x_t         = torch.sqrt(alpha_bar_t) * x0 + torch.sqrt(1 - alpha_bar_t) * eps
        return x_t, eps

diffusion = DiffusionForwardProcess(T=1000)

# -----------------------------------------------------------
# 8.2 — İleri süreç görselleştirmesi
# -----------------------------------------------------------
test_ds_viz = datasets.MNIST('./data', train=False, transform=transforms.ToTensor())
test_img, _ = test_ds_viz[0]
x0 = test_img * 2 - 1   # [0,1] → [-1,1]

timesteps_show = [0, 100, 250, 500, 750, 999]

fig, axes = plt.subplots(2, len(timesteps_show), figsize=(18, 6))
fig.suptitle('Difüzyon — İleri Süreç: Adım Adım Gürültü Ekleme', fontsize=13, fontweight='bold')

for idx, t in enumerate(timesteps_show):
    x_t, _ = diffusion.q_sample(x0, t)
    img_show = np.clip((x_t.squeeze().numpy() + 1) / 2, 0, 1)
    axes[0, idx].imshow(img_show, cmap='gray')
    axes[0, idx].set_title(f't = {t}', fontsize=11)
    axes[0, idx].axis('off')

    abar = diffusion.alpha_bar[t].item()
    axes[1, idx].bar(['Sinyal', 'Gürültü'], [abar, 1-abar],
                     color=['steelblue','salmon'], alpha=0.8)
    axes[1, idx].set_ylim(0, 1); axes[1, idx].set_ylabel('Oran')
    axes[1, idx].set_title(f'ᾱ={abar:.3f}', fontsize=9)

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(diffusion.beta.numpy(), 'b-')
axes[0].set_title('β çizelgesi (lineer)')
axes[0].set_xlabel('Zaman adımı t'); axes[0].set_ylabel('β_t'); axes[0].grid(True, alpha=0.3)
axes[1].plot(diffusion.alpha_bar.numpy(), 'g-')
axes[1].set_title("ᾱ_t = Π(1-β_s)\nt=1000'de ≈ 0 (saf gürültü)")
axes[1].set_xlabel('Zaman adımı t'); axes[1].set_ylabel('ᾱ_t'); axes[1].grid(True, alpha=0.3)
plt.suptitle('Difüzyon Parametreleri', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------
# 8.3 — Gürültü Tahmin Ağı (basit U-Net benzeri, skip bağlantılı)
# -----------------------------------------------------------
class SimpleUNet(nn.Module):
    """Basitleştirilmiş gürültü tahmincisi: eps_theta(x_t, t)"""
    def __init__(self, img_size=28, T=1000):
        super().__init__()
        self.T = T
        # Zaman gömüşü: t → 64 boyutlu vektör
        self.time_emb = nn.Embedding(T, 64)

        # Encoder
        self.enc1 = nn.Sequential(nn.Linear(784 + 64, 512), nn.SiLU())
        self.enc2 = nn.Sequential(nn.Linear(512, 256),       nn.SiLU())

        # Bottleneck
        self.bottleneck = nn.Sequential(nn.Linear(256, 256), nn.SiLU())

        # Decoder — skip connections ile (encoder çıkışları eklenir)
        self.dec1 = nn.Sequential(nn.Linear(256 + 256, 512), nn.SiLU())
        self.dec2 = nn.Sequential(nn.Linear(512 + 512, 784), nn.Identity())

    def forward(self, x, t):
        t_emb = self.time_emb(t)               # (B, 64)
        h0    = torch.cat([x, t_emb], dim=1)   # (B, 784+64)

        h1 = self.enc1(h0)                     # (B, 512)
        h2 = self.enc2(h1)                     # (B, 256)
        h3 = self.bottleneck(h2)               # (B, 256)

        h4  = self.dec1(torch.cat([h3, h2], dim=1))   # skip: h2
        out = self.dec2(torch.cat([h4, h1], dim=1))   # skip: h1
        return out

ddpm_net   = SimpleUNet(T=1000).to(device)
opt_ddpm   = optim.Adam(ddpm_net.parameters(), lr=2e-4)
params_cnt = sum(p.numel() for p in ddpm_net.parameters())
print(f'DDPM Gürültü Ağı — Parametre sayısı: {params_cnt:,}')

# -----------------------------------------------------------
# 8.4 — DDPM Eğitimi
# -----------------------------------------------------------
DDPM_EPOCHS = 5
ddpm_losses = []

for epoch in range(DDPM_EPOCHS):
    ddpm_net.train()
    epoch_loss = 0
    for x_batch, _ in train_loader:
        batch = x_batch.size(0)
        x0_b  = (x_batch.view(batch, -1) * 2 - 1).to(device)  # [0,1]→[-1,1]

        t           = torch.randint(0, 1000, (batch,)).to(device)
        alpha_bar_t = diffusion.alpha_bar[t.cpu()].to(device).view(-1, 1)
        eps_true    = torch.randn_like(x0_b)
        x_t_b       = torch.sqrt(alpha_bar_t)*x0_b + torch.sqrt(1-alpha_bar_t)*eps_true

        eps_pred   = ddpm_net(x_t_b, t)
        loss_ddpm  = F.mse_loss(eps_pred, eps_true)

        opt_ddpm.zero_grad(); loss_ddpm.backward(); opt_ddpm.step()
        epoch_loss += loss_ddpm.item()

    avg = epoch_loss / len(train_loader)
    ddpm_losses.append(avg)
    print(f'Epoch {epoch+1}/{DDPM_EPOCHS} | MSE Kaybı = {avg:.5f}')

print('\nDDPM eğitimi tamamlandı! ✅')

In [ ]:
# -----------------------------------------------------------
# 8.5 — DDPM Ters Süreç: Örnekleme
# DÜZELTME: Kullanılmayan 'trajectory' listesi kaldırıldı
# -----------------------------------------------------------
@torch.no_grad()
def ddpm_sample(model, diffusion, n_samples=16, img_size=784, device='cpu'):
    """Saf gürültüden başlayarak ters süreci (t=T → t=0) uygula."""
    model.eval()
    x         = torch.randn(n_samples, img_size).to(device)   # x_T ~ N(0,I)
    keyframes = []   # Belirli adımlardaki ara görüntüler

    for t_val in tqdm(range(999, -1, -1), desc='Örnekleme', leave=False):
        t_tensor    = torch.full((n_samples,), t_val, dtype=torch.long).to(device)
        eps_pred    = model(x, t_tensor)

        alpha_t     = diffusion.alpha[t_val].to(device)
        alpha_bar_t = diffusion.alpha_bar[t_val].to(device)

        # x0 tahmini (clamp ile sayısal kararlılık)
        x0_pred = (x - torch.sqrt(1 - alpha_bar_t)*eps_pred) / torch.sqrt(alpha_bar_t)
        x0_pred = x0_pred.clamp(-1, 1)

        if t_val > 0:
            alpha_bar_prev = diffusion.alpha_bar[t_val - 1].to(device)
            beta_t = diffusion.beta[t_val].to(device)
            mu     = ((torch.sqrt(alpha_bar_prev)*beta_t / (1 - alpha_bar_t)) * x0_pred +
                      (torch.sqrt(alpha_t)*(1 - alpha_bar_prev) / (1 - alpha_bar_t)) * x)
            sigma  = torch.sqrt(beta_t * (1 - alpha_bar_prev) / (1 - alpha_bar_t))
            x      = mu + sigma * torch.randn_like(x)
        else:
            x = x0_pred

        if t_val in [999, 750, 500, 250, 100, 0]:
            keyframes.append((t_val, x.cpu().clone()))

    return x.cpu(), keyframes

samples_ddpm, keyframes = ddpm_sample(ddpm_net, diffusion, n_samples=16, device=device)

In [ ]:
# -----------------------------------------------------------
# 8.6 — DDPM Görselleştirme
# DÜZELTME: add_axes/tight_layout çakışması giderildi.
#   Ters süreç + kayıp: üst satır (subplots)
#   Üretilen grid:      ayrı figure
# -----------------------------------------------------------
n_kf = len(keyframes)   # 6

fig, axes = plt.subplots(1, n_kf + 1, figsize=(20, 3))
fig.suptitle('DDPM — Ters Süreç & Eğitim Kaybı', fontsize=13, fontweight='bold')

# Ters süreç: t=999 → t=0 (soldan sağa)
for col, (t_val, imgs) in enumerate(reversed(keyframes)):
    img = np.clip((imgs[0].view(28, 28).numpy() + 1) / 2, 0, 1)
    axes[col].imshow(img, cmap='gray')
    axes[col].set_title(f't={t_val}', fontsize=10)
    axes[col].axis('off')

# Kayıp eğrisi
axes[-1].plot(ddpm_losses, 'purple', marker='o', ms=6)
axes[-1].set_xlabel('Epoch'); axes[-1].set_ylabel('MSE')
axes[-1].set_title('Eğitim Kaybı')
axes[-1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Üretilen görüntüler — ayrı figure (add_axes çakışmasını önler)
final_imgs   = np.clip((samples_ddpm.view(-1, 28, 28).numpy() + 1) / 2, 0, 1)
canvas_ddpm  = np.zeros((4*28, 4*28))
for idx in range(16):
    r, c = idx // 4, idx % 4
    canvas_ddpm[r*28:(r+1)*28, c*28:(c+1)*28] = final_imgs[idx]

fig2, ax2 = plt.subplots(figsize=(6, 6))
ax2.imshow(canvas_ddpm, cmap='gray')
ax2.set_title('DDPM — Üretilen Görüntüler (4×4)', fontsize=12, fontweight='bold')
ax2.axis('off')
plt.tight_layout()
plt.show()

---
## Bölüm 9: Model Karşılaştırması & FID Metriği

**Fréchet Inception Distance (FID):**
$$\text{FID} = \|\mu_r - \mu_g\|^2 + \text{Tr}\left(\Sigma_r + \Sigma_g - 2\sqrt{\Sigma_r\Sigma_g}\right)$$

Düşük FID = Daha iyi model!

In [ ]:
# -----------------------------------------------------------
# 9.1 — FID Hesabı (özellik uzayında)
# -----------------------------------------------------------
def compute_fid(real_features, gen_features):
    """
    FID = ||mu_r - mu_g||^2 + Tr(Sigma_r + Sigma_g - 2*sqrt(Sigma_r @ Sigma_g))
    Gerçek uygulamada: Inception v3 feature'ları kullanılır.
    Bu demoda: PCA ile indirgenmiş piksel feature'ları.
    """
    mu_r    = np.mean(real_features, axis=0)
    mu_g    = np.mean(gen_features,  axis=0)
    Sigma_r = np.cov(real_features, rowvar=False)
    Sigma_g = np.cov(gen_features,  rowvar=False)

    covmean, _ = linalg.sqrtm(Sigma_r @ Sigma_g, disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real   # sayısal hatalar için güvenlik

    fid = (np.sum((mu_r - mu_g)**2) +
           np.trace(Sigma_r + Sigma_g - 2*covmean))
    return float(fid)

# -----------------------------------------------------------
# 9.2 — Modellerin FID skorunu tahmin et (PCA feature uzayında)
# -----------------------------------------------------------
real_imgs_np = x_test[:500].view(-1, 784).numpy()
pca_fid      = PCA(n_components=50).fit(real_imgs_np)
real_feats   = pca_fid.transform(real_imgs_np)

# VAE örnekleri
vae.eval()
with torch.no_grad():
    z_v       = torch.randn(500, LATENT_DIM).to(device)
    vae_imgs  = vae.decode(z_v).cpu().numpy()
vae_feats = pca_fid.transform(vae_imgs)

# GAN örnekleri
G.eval()
with torch.no_grad():
    z_g      = torch.randn(500, Z_DIM).to(device)
    gan_imgs = ((G(z_g) + 1) / 2).cpu().numpy()
gan_feats = pca_fid.transform(gan_imgs)

# DDPM örnekleri
# DÜZELTME: np.tile ile 16 görüntüyü tekrarlama kaldırıldı — istatistik yanıltıcıydı.
# Bunun yerine 64 örnek üretilir; FID için hâlâ sınırlı ama dürüst bir tahmin.
print('DDPM için ek örnekler üretiliyor (64 adet)...')
with torch.no_grad():
    ddpm_extra, _ = ddpm_sample(ddpm_net, diffusion, n_samples=64, device=device)
ddpm_imgs_np = np.clip((ddpm_extra.view(-1, 784).numpy() + 1) / 2, 0, 1)
# Gerçek veri sayısına eşleştirmek için tile (64→500); notda belirtilir
ddpm_imgs_500 = np.tile(ddpm_imgs_np, (8, 1))[:500]
ddpm_feats    = pca_fid.transform(ddpm_imgs_500)

real_feats_a = real_feats[:250]
real_feats_b = real_feats[250:]

fid_ideal = compute_fid(real_feats_a, real_feats_b)
fid_vae   = compute_fid(real_feats, vae_feats)
fid_gan   = compute_fid(real_feats, gan_feats)
fid_ddpm  = compute_fid(real_feats, ddpm_feats)

print('FID Skorları (düşük = iyi | PCA feature uzayı | sınırlı epoch):')
print(f'  İdeal (gerçek vs gerçek): {fid_ideal:.2f}')
print(f'  VAE:   {fid_vae:.2f}')
print(f'  GAN:   {fid_gan:.2f}')
print(f'  DDPM:  {fid_ddpm:.2f}  (Not: 64 benzersiz örnek tile edildi — yaklaşık değer)')
print()
print('⚠️  Uyarı: Gerçek FID, Inception v3 ile ~50k örnekte hesaplanır.')
print('    Bu demo eğitici amaçlıdır; sayısal karşılaştırma görecelidir.')

In [ ]:
# -----------------------------------------------------------
# 9.3 — Kapsamlı Model Karşılaştırması
# -----------------------------------------------------------
fig = plt.figure(figsize=(18, 12))
fig.suptitle('Üretken Modeller — Kapsamlı Karşılaştırma', fontsize=15, fontweight='bold')

# ── FID Karşılaştırması ─────────────────────────────────────
ax1 = fig.add_subplot(2, 3, 1)
models_list = ['İdeal', 'VAE', 'GAN', 'DDPM']
fid_scores  = [fid_ideal, fid_vae, fid_gan, fid_ddpm]
bar_colors  = ['green', 'steelblue', 'salmon', 'purple']
bars = ax1.bar(models_list, fid_scores, color=bar_colors, alpha=0.8, edgecolor='black')
for bar, val in zip(bars, fid_scores):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'{val:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax1.set_ylabel('FID Skoru (↓ daha iyi)')
ax1.set_title('FID Karşılaştırması\n(PCA Feature Uzayı, demo)')
ax1.grid(True, alpha=0.3, axis='y')

# ── Radar grafik ────────────────────────────────────────────
ax2 = fig.add_subplot(2, 3, 2, polar=True)
categories   = ['Kalite', 'Çeşitlilik', 'Hız', 'Kararlılık', 'Latent\nKontrol']
model_scores = {
    'VAE':      [2, 3, 5, 5, 5],
    'GAN':      [5, 4, 5, 2, 3],
    'GMMN':     [2, 3, 4, 4, 2],
    'Difüzyon': [5, 5, 1, 5, 4],
}
N      = len(categories)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist() + [0]
radar_colors = ['blue','red','green','purple']
for (model_name, scores), color in zip(model_scores.items(), radar_colors):
    values = scores + scores[:1]
    ax2.plot(angles, values, 'o-', lw=2, color=color, label=model_name)
    ax2.fill(angles, values, alpha=0.1, color=color)
ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(categories, size=9)
ax2.set_ylim(0, 5)
ax2.set_title('Çok Boyutlu Karşılaştırma\n(5=En iyi)', pad=20)
ax2.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

# ── Kronoloji ───────────────────────────────────────────────
ax3 = fig.add_subplot(2, 3, 3)
timeline = {
    1985:'RBM', 2006:'DBN', 2013:'VAE', 2014:'GAN',
    2015:'GMMN', 2017:'WGAN', 2019:'StyleGAN',
    2020:'DDPM', 2022:'Stable\nDiff.'
}
years    = list(timeline.keys())
models_t = list(timeline.values())
t_colors = ['gray','gray','steelblue','salmon','green',
            'orange','darkorange','purple','darkviolet']
ax3.scatter(years, [1]*len(years), c=t_colors, s=150, zorder=5)
ax3.plot(years, [1]*len(years), 'k-', alpha=0.3, zorder=4)
for yr, mdl, c in zip(years, models_t, t_colors):
    ax3.annotate(f'{yr}\n{mdl}', (yr, 1),
                 textcoords='offset points', xytext=(0, 15),
                 ha='center', fontsize=8, color=c, fontweight='bold',
                 arrowprops=dict(arrowstyle='-', color=c, lw=0.8))
ax3.set_xlim(1982, 2024); ax3.set_ylim(0.8, 1.5)
ax3.set_yticks([]); ax3.set_title('Üretken Modeller Kronolojisi')
ax3.set_xlabel('Yıl'); ax3.grid(True, alpha=0.2, axis='x')

# ── Karşılaştırma Tablosu ───────────────────────────────────
ax4 = fig.add_subplot(2, 1, 2)
ax4.axis('off')
table_data = [
    ['Özellik',            'VAE',            'GAN',          'GMMN',         'Difüzyon'],
    ['Teorik Temel',       'ELBO/KL',        'Nash Dengesi', 'MMD/Kernel',   'Score Matching'],
    ['Latent Uzay',        'Açık (ELBO)',    'Örtük',        'Örtük (MMD)',  'Markov Zinciri'],
    ['Örnek Kalitesi',     '★★☆☆☆',         '★★★★☆',       '★★☆☆☆',       '★★★★★'],
    ['Eğitim Kararlılığı', '★★★★☆',         '★★☆☆☆',       '★★★★☆',       '★★★★★'],
    ['Örnekleme Hızı',     '★★★★★',         '★★★★★',       '★★★★☆',       '★☆☆☆☆'],
    ['Mod Çöküşü',         'Düşük',         'Yüksek',       'Düşük',        'Çok Düşük'],
    ['Bulanıklık',         'Var',           'Yok',          'Orta',         'Yok'],
    ['Temsil Modelleri',   'β-VAE',         'StyleGAN',     'GMMN',         'DDPM, SD'],
]

table = ax4.table(
    cellText=table_data[1:], colLabels=table_data[0],
    cellLoc='center', loc='center', bbox=[0, 0, 1, 1]
)
table.auto_set_font_size(False)
table.set_fontsize(10)
for j in range(5):
    table[0,j].set_facecolor('#2C3E50')
    table[0,j].set_text_props(color='white', fontweight='bold')
row_colors = ['#EAF2F8','#FDFEFE']
for i in range(1, len(table_data)):
    for j in range(5):
        table[i,j].set_facecolor(row_colors[i%2])
ax4.set_title('Kapsamlı Model Karşılaştırma Tablosu', fontweight='bold', pad=10)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

---
## 📝 Ders Özeti & Kilit Çıkarımlar

| Konu | Kilit Formül | Pratik Anlam |
|------|-------------|---------------|
| MLE | $\theta^* = \arg\max \mathbb{E}[\log p_\theta(x)]$ | Veriyi olasılıklı hale getir |
| KL | $D_{KL}(p\|q) \geq 0$ | MLE = KL minimizasyonu |
| ELBO | $\log p(x) \geq \mathcal{L}(x)$ | Intractable p(x) için alt sınır |
| VAE | $z = \mu + \sigma \odot \varepsilon$ | Reparametrization → backprop |
| GAN | $\min_G \max_D V(G,D)$ | Rekabet ile öğrenme |
| MMD | $\mathbb{E}[k(x,x')] - 2\mathbb{E}[k(x,y)] + \mathbb{E}[k(y,y')]$ | Kernel ile dağılım mesafesi |
| DDPM | $\mathcal{L} = \mathbb{E}\|\varepsilon - \varepsilon_\theta(x_t,t)\|^2$ | Gürültüyü tahmin et |
| FID | $\|\mu_r-\mu_g\|^2 + \text{Tr}(...)$ | Kalite metriği (↓ daha iyi) |

In [ ]:
print('='*60)
print('🎓 Üretken Yapay Zeka — 3. Ders Notebook Tamamlandı!')
print('='*60)
print()
print("Bu notebook'ta öğrendiklerimiz:")
print('  ✅ MLE ve KL Iraksaması — olasılıksal çerçeve')
print("  ✅ Manifold Hipotezi — MNIST'te 784D → ~10D")
print('  ✅ ELBO türetimi ve bileşenleri')
print('  ✅ VAE: Reparametrization Trick + β-VAE')
print('  ✅ GAN: Minimax oyunu + Nash dengesi')
print('  ✅ GMMN: MMD kernel yaklaşımı')
print('  ✅ DDPM: İleri/ters süreç + gürültü tahmini')
print('  ✅ FID metriği ile model değerlendirme')
print()
print('Sonraki ders: Transformer tabanlı üretken modeller!')